# Inside the one call

**`resolve_documents()` in three cells is the product. This is what it does.**

[Notebook 02](02_same_person_across_documents.ipynb) turns three PDFs into
verdicts in one line. That is the right default, and it is also a black box. If
you are going to act on a decision — merge two customer records, pay an invoice,
file a report — you need to see what the box did.

So this notebook opens it. Same three documents, one layer per cell: parse,
detect, extract, assemble, compare, decide, attest.

Two things you will find on the way, neither of which is in the marketing:

* the pipeline confidently flags **36 Nigerian tax identification numbers** in a
  British bank statement, and every one is wrong;
* two documents score **0.9974** and are still not merged.

Both are the system working. The second one is the whole thesis.


## 1. Parse — a PDF is not a string

`doc.parse` returns a structured document, not text. The distinction matters
because a bank statement *is* a table: read it as a flat string and every row
runs into the next, and the address on line 3 becomes part of the transaction on
line 4.


In [1]:
import os, glob, logging, warnings
from collections import Counter
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
logging.disable(logging.INFO); warnings.filterwarnings("ignore")

from arche.doc import parse

DOCS = sorted(glob.glob("../../data/docs/*.pdf"))
doc = parse(DOCS[0])
print(len(DOCS), "documents")
print()
print("available:", [a for a in dir(doc) if not a.startswith("_")])
print(f"pages {doc.num_pages}   text {len(doc.text):,} chars   tables {len(doc.tables)}")
print()
for line in doc.text.strip().splitlines()[:4]:
    print("  ", line[:78])


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


3 documents

available: ['info', 'json', 'markdown', 'metadata', 'num_pages', 'source', 'tables', 'text']
pages 10   text 28,906 chars   tables 8

   Dennis Aibuedefe Irorere
   
   Apartment 2 Coal House 3 Marina Place Birmingham B16 8WS United Kingdom
   


`markdown` and `json` preserve the layout, `tables` gives the tabular regions
separately. Everything downstream reads `text`, but the structure is there when
a field only makes sense in its cell.

## 2. Detect — what counts as personal data depends on where you are

The same bytes, two jurisdictions. This is not a formatting difference: a
statute pack decides which categories exist, what tier they sit in, and which
clause to cite.


In [2]:
from arche import Pipeline

text = doc.text
for jurisdiction in ("NG", "GDPR"):
    result = Pipeline(jurisdiction=jurisdiction).process(text)
    print(f"{jurisdiction:6} {dict(Counter(d.category for d in result.detections))}")

result = Pipeline(jurisdiction="NG").process(text)
d = result.detections[0]
print()
print("one detection carries:")
for attr in ("category", "confidence", "sensitivity_tier", "regulatory_citation",
             "identity_class", "detector"):
    print(f"  {attr:22} {getattr(d, attr, None)}")


NG     {'PII-2-TIN': 36, 'PII-4-ADDRESS': 11, 'PII-3-PHONE': 1}
GDPR   {'PII-4-ADDRESS': 11, 'PII-3-PHONE': 1}

one detection carries:
  category               PII-2-TIN
  confidence             0.55
  sensitivity_tier       moderate
  regulatory_citation    NDPA-2023 s.30, FIRS guidelines
  identity_class         functional
  detector               rule:ng_tin


`PII-2-TIN` exists under `NG` and not under `GDPR` — a Nigerian tax
identification number is a named category in the NDPA and simply is not a thing
the GDPR pack looks for. The detection cites `NDPA-2023 s.30` because a reviewer
asking "why was this redacted?" deserves a clause, not a category name.

## 3. The 36 tax numbers that are not tax numbers

Thirty-six TIN detections in a British bank statement should bother you. Look at
what they actually matched.


In [3]:
tins = [d for d in result.detections if d.category.endswith("TIN")]
print(f"{len(tins)} detections, {len({t.text for t in tins})} distinct values\n")
for t in tins[:5]:
    context = text[max(0, t.start - 34):t.end + 16].replace("\n", " ")
    print(f"  {t.text:12} conf {t.confidence:.2f}   ...{context}...")


36 detections, 22 distinct values

  1747722393   conf 0.55   ...05.72 | | 19/02/2026 | VIATOR *IT-1747722393 London GBR     ...
  5063127134   conf 0.55   ...CITY CO (Direct Debit) Reference: 5063127134                ...
  2601272201   conf 0.55   ...356.92 | | 27/01/2026 | BOLT.EU/O/2601272201 London GBR This...
  2601272203   conf 0.55   ...517.92 | | 27/01/2026 | BOLT.EU/O/2601272203 London GBR     ...
  2601272200   conf 0.55   ...-------| | 27/01/2026 | BOLT.EU/O/2601272200 London GBR This...


Bolt ride references. Viator transaction IDs. A direct-debit reference. Not one
is a tax number.

Three things are true at once here, and it is worth separating them:

1. **The detector is doing its job.** A Nigerian TIN is ten digits; these are ten
   digits. Without context there is no way to tell them apart on shape alone.
2. **The confidence says so.** Every one lands at **0.55**, not 0.95. The number
   is not decoration — it is the detector declining to be certain.
3. **None of them reached the record.** The record builder consumes only the
   identifier categories it maps to canonical fields, and `TIN` is not one of
   them. Thirty-six false positives, zero contamination of the match.

That third point is the design, not luck. **Detection and resolution are
separate boundaries.** A detector may over-fire — for redaction, over-firing is
the safe direction — without that noise ever becoming identity evidence. If TIN
*were* mapped, these 22 distinct values would be sitting in the record as
national identifiers, and a shared Bolt reference would look like a shared ID.

**This is now caught automatically.** `resolve_documents` infers the
jurisdiction per document, and this statement's own evidence — UK postcodes, a
sort code, `Registered in England and Wales` — names GB with no ambiguity. The
run above passes `jurisdiction="NG"` explicitly so the failure is visible; on
the default (`jurisdiction="auto"`) these 36 detections do not occur, and an
explicit code that disagrees with the document is recorded in
`report.jurisdiction_conflicts` rather than left silent.

The fix took two pieces, not one. Detecting GB alone would have taken the error
count from 36 to zero **by switching redaction off**, because no UK statute pack
existed and a Pipeline with no statute returns text unchanged. A UK pack and a
conservative floor had to land first. Measured: 36 -> 0 false detections, and
the document is still redacted.

## 4. Extract — recognised, not validated

Detectors find things with *structure* — an email, a phone number, a checksum.
The extractor finds things that only a model can see: a person, an organisation,
a place. Two different jobs, deliberately not merged.


In [4]:
from arche.extract import extract

entities = list(extract(text[:3000]))
print(dict(Counter(str(e.entity_type) for e in entities)))
print()
for e in sorted(entities, key=lambda e: -e.confidence)[:6]:
    print(f"  {str(e.entity_type):14} {e.text[:38]:40} conf {e.confidence:.3f}")


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'PERSON': 1, 'NATIONAL_ID': 2, 'PHONE': 1, 'DATE': 12, 'MONEY': 5, 'ORGANIZATION': 2, 'URL': 1, 'LOCATION': 1}

  URL            https://monzo.com)                       conf 0.950
  ORGANIZATION   Monzo Bank Limited                       conf 0.942
  LOCATION       Broadwalk House, 5 Appold Street, Lond   conf 0.917
  PERSON         Dennis Aibuedefe Irorere                 conf 0.845
  DATE           28/02/2026                               conf 0.803
  DATE           01/12/2025                               conf 0.800


The split is load-bearing. An identifier is **validated** — check digits, length,
prefix — so a match on one is strong evidence. A name is **recognised**, with a
confidence attached, so a match on one is weaker evidence that has to be weighed
against how common the name is.

Collapse the two and you get the failure this whole project exists to prevent: a
pattern-matched "identifier" that is really a formatting coincidence, treated as
proof that two people are one person.

## 5. Assemble — three documents, three records


In [5]:
from arche import resolve_documents

report = resolve_documents("../../data/docs/*.pdf", jurisdiction="NG")
for doc_name, rec in report.records.items():
    print(f"{doc_name[:40]:42}")
    for k, v in sorted(rec.items()):
        print(f"    {k:14} {v}")
    print()


[0/3] resolving 3 document(s) (0.0s)


[1/3] parsing Emailing Monzo_bank_statement_2025-12-01-2026-02-28_2231.pdf (0.0s)


[1/3] detecting + extracting Emailing Monzo_bank_statement_2025-12-01-2026-02-28_2231.pdf (76.3s)


[2/3] parsing Invoice-PEDHCF-00012.pdf (79.1s)


[2/3] detecting + extracting Invoice-PEDHCF-00012.pdf (92.5s)


[3/3] parsing Paystatement_2025-12-23T00_00_00.pdf (94.7s)


[3/3] detecting + extracting Paystatement_2025-12-23T00_00_00.pdf (116.9s)


[0/3] comparing records (119.4s)


[0/3] 3 record(s), 3 verdict(s) (119.8s)


Emailing Monzo_bank_statement_2025-12-01  
    address        3 Marina Place
    name           Dennis Aibuedefe Irorere
    organisation   Monzo Bank Limited
    phone          72111117

Invoice-PEDHCF-00012.pdf                  
    address        PO BOX 7775
    email          denironyx@gmail.com
    name           Dennis Irorere
    organisation   Netlify, Inc.

Paystatement_2025-12-23T00_00_00.pdf      
    address        3 Marina Place, Birmingham B16
    name           Dennis Irorere
    organisation   Viator Ltd



Read the names.

```
Monzo statement  ->  Dennis Aibuedefe Irorere
Invoice          ->  Dennis Irorere
Payslip          ->  Dennis Irorere
```

One document carries a middle name and two do not. That is the single most
ordinary fact in identity data, and it is about to decide everything.

The addresses vary too — `3 Marina Place`, `3 Marina Place, Birmingham B16`, and
a `PO BOX` that belongs to the *issuer* rather than the person. Nobody wrote a
rule for any of this.

## 6. Compare — the factors behind each score


In [6]:
for dec in report.decisions:
    print(f"{dec['a'][:24]:26} vs {dec['b'][:24]:26} -> {dec['score']:.4f}")
    for k, v in sorted(dec["factors"].items()):
        print(f"     {k:12} {v}")
    print()


Emailing Monzo_bank_stat   vs Invoice-PEDHCF-00012.pdf   -> 0.9656
     address      0.4416
     name         0.8
     name_tf      0.6393

Emailing Monzo_bank_stat   vs Paystatement_2025-12-23T   -> 0.9974
     address      1.0
     name         0.8
     name_tf      0.6393

Invoice-PEDHCF-00012.pdf   vs Paystatement_2025-12-23T   -> 0.9903
     address      0.4157
     name         1.0
     name_tf      1.0



`name` is string similarity. `name_tf` is the same comparison **weighted by how
distinctive the shared tokens are** — a match on `Irorere` is worth far more than
a match on a common given name, because rarity is what identifies.

The middle name costs the Monzo pairs on both: `name` 0.8 instead of 1.0, and
`name_tf` **0.6393** instead of 1.0.

## 7. Decide — why 0.9974 is not a merge

Here is the result that matters.


In [7]:
from arche.resolve._gate import DISTINCTIVE_FLOOR

print(f"gate floor = {DISTINCTIVE_FLOOR}\n")
print(f"{'pair':<50} {'score':>7}  {'name_tf':>8}  verdict")
print("-" * 84)
for dec in report.decisions:
    pair = f"{dec['a'][:22]} / {dec['b'][:22]}"
    tf = dec["factors"].get("name_tf", 0.0)
    print(f"{pair:<50} {dec['score']:>7.4f}  {tf:>8.4f}  {dec['identity']}")


gate floor = 0.75

pair                                                 score   name_tf  verdict
------------------------------------------------------------------------------------
Emailing Monzo_bank_st / Invoice-PEDHCF-00012.p     0.9656    0.6393  review
Emailing Monzo_bank_st / Paystatement_2025-12-2     0.9974    0.6393  review
Invoice-PEDHCF-00012.p / Paystatement_2025-12-2     0.9903    1.0000  same_entity


**The highest-scoring pair in the notebook is not a match.**

`Monzo / Payslip` scores **0.9974** and returns `review`. `Invoice / Payslip`
scores **0.9903** — lower — and returns `same_entity`. The ordering by score is
the opposite of the ordering by verdict.

The reason is the gate. A merge requires a *distinctive* signal above **0.75**,
and the Monzo pairs sit at `name_tf` 0.6393 because `Dennis Aibuedefe Irorere`
and `Dennis Irorere` do not share enough rare material. The score says the
records are broadly consistent. The gate says nothing rare enough has been
agreed on to justify merging two people's financial records without a human.

**A score is not a decision.** This is the same rule that stops two
`General Hospital` records merging on an identical name, and the same one that
stops two people called `Ibrahim Musa` becoming one person. It is not tuned for
this notebook; it is the shipped constant, and lowering it to 0.70 to make this
pair merge would break that person case — measured, not assumed.

The honest reading of this run: **one confirmed match, two cases for a human**,
from three documents with no shared identifier.

## 8. Attest — a decision you can cite


In [8]:
dec = report.decisions[0]
print("decision_id:", dec["decision_id"])
print()
print("A content hash over the evidence and the pins — no timestamp, no")
print("randomness. Anyone holding the same inputs recomputes the same id,")
print("which is what makes the verdict checkable rather than merely stored.")
print()
report.save_json("../../data/docs/report.json")
print("full report written to data/docs/report.json")
print()
print("Note it is written INSIDE data/docs/, which is gitignored. Values are")
print("masked by default, but a report derived from personal documents still")
print("names the bank, the employer and the statement dates in its filenames.")


decision_id: dec:sha256:6905b79403b22a17dc471dd2d054882a30ba314c7230e3af9845b27a6d146238

A content hash over the evidence and the pins — no timestamp, no
randomness. Anyone holding the same inputs recomputes the same id,
which is what makes the verdict checkable rather than merely stored.

full report written to data/docs/report.json

Note it is written INSIDE data/docs/, which is gitignored. Values are
masked by default, but a report derived from personal documents still
names the bank, the employer and the statement dates in its filenames.


## What this establishes, and what it does not

**Establishes.** Every layer is inspectable, and the interesting behaviour is
visible rather than buried: a jurisdiction changing what counts as personal
data, a detector over-firing without contaminating the match, and a gate
refusing a 0.9974 merge for a stated reason.

**Does not establish.**

* **Three documents is a demonstration.** For measured accuracy on data we
  neither chose nor labelled, see the [false-merge-rate](06_what_is_the_false_merge_rate.ipynb)
  and [places benchmark](07_places_on_a_public_benchmark.ipynb) notebooks.
* **Extraction bounds everything downstream.** If the extractor picks the wrong
  span for a name, resolution is judging the wrong string with full confidence.
  Cell 4 exists so you look at that before trusting cell 7.
* **The `PO BOX` in the invoice record is the issuer's address, not the
  person's.** Nothing in this pipeline knows the difference, and it quietly
  weakened one comparison. Field-level provenance — *whose* address is this? —
  is not solved here.
* **36 false positives is a real false-positive rate**, on one document, from
  one detector. It was harmless because of how the record is assembled. Map TIN
  into the record and it stops being harmless.
* **No OCR.** A scanned image yields no text and lands in `report.errors`.
